In [18]:
def generate_f(n, o, q):
    '''
    generates a random quadratic equation over F_q in n variables.
    q has to be odd.
    '''
    F = GF(q)
    # define the polynomial ring where the variables live
    R = PolynomialRing(F, n, 't')
    # construct a quadratic polynomial over F_q, such that every term has a vinegar variable
    f_k = sum(R.gens()[i]*sum(F.random_element()*R.gens()[j] for j in range(n)) for i in range(o, n)) 
    print(type(f_k))
    # print(f_k.coefficients())
    size = len(R.gens())
    gram = Matrix(F, size, size)
    for i in range(size):
        quadratic_terms = {R.gens()[i]: 2}
        gram[i, i] = f_k.coefficient(quadratic_terms)
        for j in range(i+1, n):
            other_terms = {R.gens()[i]: 1, R.gens()[j]: 1}
            gram[i, j] = f_k.coefficient(other_terms)
    print(gram)
    

    # Q = QuadraticForm(f_k);
    # # gets matrix of polar form
    # gram = Q.matrix() * F(2)**(-1);
    
    # further checks
    x = vector(R.gens())
    assert (x * gram * x == f_k)
    return gram

generate_f(8, 4, 9)

<class 'sage.rings.polynomial.multi_polynomial_libsingular.MPolynomial_libsingular'>
[2, 1, 2, z2 + 2, 2, z2, 2, 2*z2, z2 + 2, z2 + 2, 2*z2, z2 + 1, z2 + 2, z2 + 2, 1, 2, z2 + 2, 1, z2 + 1, z2 + 2, 2*z2 + 1, 2*z2 + 1, z2 + 2]


TypeError: not in prime subfield

In [2]:
def KeyGen(n, m, o, q):
    '''
    returns a tuple (pk, sk) containing the public key and the secret key.
    The secret is again a tuple (FF, M), where 
            - FF is the central map and 
            - M an invertible linear transformation.
    '''
    F = GF(q)
    FF = []
    # pick the quadratic equations at random
    for i in range(m):
        FF.append(generate_f(n, o, q))
    # print("FF[2] \n", FF[2])
    # choose an invertible linear transform F_n -> F_n 
    M = GL(n, F).random_element().matrix()
    sk = (FF, M)
    # construct the public key
    PP = []
    for i in range(len(FF)):
        PP.append(M.transpose()*FF[i]*M)
    # print("PP[2] \n", PP[2])
    pk = PP
    return pk, sk

In [3]:
def solve_FF(n, m, o, q, sk, t, F, R, x):
    # pick random values for the 'vinegar' variables
    for i in range(n-o):
        # print(i+o-1)
        x[i+o] = F.random_element()
    # print(x)
    # solve the system F(s') = t (m linear equations in o variables, m <= o)
    FF, M = sk
    system = [x*FF[i]*x - t[i] for i in range(m)]
    A = matrix([ [eq.coefficient(v) for v in x[0:o]] for eq in system])
    # TODO try with and without max
    b = vector([-eq.constant_coefficient() for eq in system])
    # print("het rechterlid", b)
    sol = A.solve_right(b)
    return sol

In [4]:
def sign(n, m, o, q, sk, t):
    
    '''
    returns the UOV signature as a vector of length n
    '''
    assert (len(t) == m)
    F = GF(q)
    R = PolynomialRing(F, n, 'x')
    x = vector(R.gens())
    
    # we skip the hashing part
    i = 1
    while true:
        try:
            sol_temp = solve_FF(n, m, o, q, sk, t, F, R, x)
            # print("sol", sol_temp)
            break
        except ValueError:
            i+=1
            print(f'Hélas ... Too much vinegar. Poging {i}')
    # print("de inverse transformatie", sk[1].inverse())
    # print('element', sol_temp[2])
    # create s'
    sol = [sol_temp[i] for i in range(o)]
    sol.extend([x[i] for i in range(o, n)])
    assert (len(sol) == n)
    # print(sol)
    signature = sk[1].inverse()*vector(sol)
    return signature

In [5]:
def verify(n, m, o, q, PP, signature):
    F = GF(q)
    R = PolynomialRing(F, n, 'x')
    x = vector(R.gens())
    t_bar = []
    for i in range(len(PP)):
        t_bar.append(signature*PP[i]*signature)
    return t_bar


In [6]:
# testsetup
n, m, o, q = 8, 4, 4, 9
kg = KeyGen(n, m, o, q)
pk, sk = kg[0], kg[1]
t = [GF(q).random_element() for i in range(m)]
s = sign(n, m, o, q, sk, t)
v = verify(n, m, o, q, pk, s)
# TEST
print('te ondertekenen:', t)
print('response:', v)

te ondertekenen: [z2, 2*z2, 2, 2*z2 + 2]
response: [z2, (-z2), -1, (-z2 - 1)]
